# 02 · Facts: bronze → silver → gold
All facts are built here. To add a fact: add its source to the **Silver** section, then one query and one `load_fact(...)` line to the **Gold** section.

| Gold fact | Built from |
|---|---|
| `fact_sales` | `sales` + `dim_customer` + `dim_product` |
| `fact_returns` | `returns` + the sale line it refers to (`sales`) + `dim_customer` + `dim_product` |

Run `01_dimensions` first. A fact row whose customer or product is not in the dimension (yet) gets the Unknown member `-1` and is corrected automatically on a later run, once the dimension row arrives.

In [ ]:
%pip install databricks-labs-dqx -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
%run ./00_config

In [ ]:
LANDING = LANDING_FACTS     # fact source files are read from this Volume folder

## Silver
Same call as for dimensions: columns with their types and the business key. `dq_rules.yml` also holds the **de-duplication rule** for sales (`duplicate_order_line`): of several lines with the same order, customer, product and time, the first submitted is kept.

In [ ]:
load_silver("sales", {
    "order_line_id": "bigint", "order_id": "bigint", "customer_id": "bigint", "product_id": "bigint",
    "quantity": "int", "unit_price": "decimal(12,2)", "order_ts": "timestamp", "updated_at": "timestamp"}, key="order_line_id")

load_silver("returns", {
    "return_id": "bigint", "order_line_id": "bigint", "return_ts": "timestamp", "reason": "string:lower",
    "returned_quantity": "int", "refund_amount": "decimal(12,2)", "updated_at": "timestamp"}, key="return_id")

## Gold
Each fact is a SQL query: the joins and special conditions are right there. `coalesce(..., -1)` sends a row without a matching dimension row to the Unknown member.

In [ ]:
fact_sales = spark.sql(f"""
    SELECT sl.order_line_id, sl.order_id,
           to_date(sl.order_ts)                       AS order_date,
           sl.order_ts,
           coalesce(c.customer_sk, -1)                AS customer_sk,
           coalesce(p.product_sk, -1)                 AS product_sk,
           sl.quantity, sl.unit_price,
           CAST(sl.quantity * sl.unit_price AS DECIMAL(14,2)) AS amount
    FROM {SILVER}.sales sl
    LEFT JOIN {GOLD}.dim_customer c ON c.customer_id = sl.customer_id
    LEFT JOIN {GOLD}.dim_product  p ON p.product_id  = sl.product_id
""")
load_fact("fact_sales", fact_sales, keys=["order_line_id"])

In [ ]:
# a return has no customer or product of its own: both come from the sale line it refers to
fact_returns = spark.sql(f"""
    SELECT r.return_id, r.order_line_id, sl.order_id,
           to_date(r.return_ts)                       AS return_date,
           r.return_ts,
           coalesce(c.customer_sk, -1)                AS customer_sk,
           coalesce(p.product_sk, -1)                 AS product_sk,
           r.reason, r.returned_quantity, r.refund_amount,
           datediff(to_date(r.return_ts), to_date(sl.order_ts)) AS days_to_return
    FROM {SILVER}.returns r
    LEFT JOIN {SILVER}.sales      sl ON sl.order_line_id = r.order_line_id
    LEFT JOIN {GOLD}.dim_customer c  ON c.customer_id    = sl.customer_id
    LEFT JOIN {GOLD}.dim_product  p  ON p.product_id     = sl.product_id
""")
load_fact("fact_returns", fact_returns, keys=["return_id"])

## Check the result

In [ ]:
display(dq_report().filter("source IN ('sales', 'returns')"))
display(spark.sql(f"""
    SELECT 'fact_sales' AS fact, count(*) AS rows, sum(CASE WHEN customer_sk = -1 OR product_sk = -1 THEN 1 ELSE 0 END) AS unknown_dim_key FROM {GOLD}.fact_sales
    UNION ALL
    SELECT 'fact_returns', count(*), sum(CASE WHEN customer_sk = -1 OR product_sk = -1 THEN 1 ELSE 0 END) FROM {GOLD}.fact_returns"""))
display(spark.sql(f"""
    SELECT f.order_date, c.name, p.product_name, f.quantity, f.amount
    FROM {GOLD}.fact_sales f JOIN {GOLD}.dim_customer c USING (customer_sk) JOIN {GOLD}.dim_product p USING (product_sk)
    ORDER BY f.order_ts DESC LIMIT 10"""))

### Rejected duplicates
Every row below was removed by the `duplicate_order_line` rule; `kept_line` is the first-submitted line that survived.

In [ ]:
display(spark.sql(f"""
    SELECT q.order_line_id AS rejected_line, s.order_line_id AS kept_line, q.order_id, q.customer_id, q.product_id, q.order_ts
    FROM {SILVER}.sales_quarantine q
    JOIN {SILVER}.sales s ON s.order_id = CAST(q.order_id AS BIGINT) AND s.customer_id = CAST(q.customer_id AS BIGINT)
                         AND s.product_id = CAST(q.product_id AS BIGINT) AND s.order_ts = CAST(q.order_ts AS TIMESTAMP)
    WHERE array_contains(q._errors, 'duplicate_order_line')
    ORDER BY kept_line, rejected_line"""))